Miscanthus Crop Parameter File - Derivation & Documentation
-----------------------------------------------------------------------------------------------------------------------------------------------
In this Notebook the process towards the generation of a Miscanthus crop parameter file is documented. This is done so that all derivations and assumptions are clearly documented, and can be reproduced.

The outcome of this Notebook is the generation of a readable crop parameter .yaml file by the YAMLCropDataProvider function in PCSE.

In [24]:
import numpy as np
import pandas as pd
from pathlib import Path
import yaml
import os 

In [25]:
# Create output directory
folder = Path("/Users/paulianoprescu/Projects/wofost_miscanthus/miscanthus_calibration")
output_dir = folder / "input_params" / "miscanthus"

In [26]:
# Defining Generic C3 & C4 parameters 
# Firstly we will define the generic C3 parameters. 
# This step is rather a formality, since Miscanthus is a C4 plant, which means that the C3 parameters will not be used in the WOFOST model.

# Multiplication factor for EFF to account for an increasing CO2 concentration in PPM
CO2EFFTB_C3 = [40., 0.00, 360., 1.00, 720., 1.11, 1000., 1.11, 2000., 1.11]
CO2EFFTB_C4 = [40., 0.00, 360., 1.00, 720., 1.00, 1000., 1.00, 2000., 1.00]

# Multiplication factor for maxmimum transpiration rate TRAMX to account for an increasing CO2 concenctration in PPM
CO2TRATB_C3 = [40., 0.00, 360., 1.00, 720., 0.9, 1000., 0.9, 2000., 0.9]
CO2TRATB_C4 = [40., 0.00, 360., 1.00, 720., 0.74, 1000., 0.74, 2000., 0.74]

# Multiplication factor for AMAX to account for an increasing CO2 concentration in PPM
CO2AMAXTB_C3 = [40., 0.00, 360., 1.00, 720., 1.60, 1000., 1.90, 2000., 1.90]
CO2AMAXTB_C4 = [40., 0.00, 360., 1.00, 720., 1.00, 1000., 1.00, 2000., 1.00]


In [27]:
# Emergence parameters
#
# Lower Threshold Temperature for Emergence (TBASEM) in degrees Celsius
# Taken from AquaCrop basetemperature, filled in by Francesco Pancaldi & Luisa Trindade (WUR)
TBASEM = 8.0

# Maximum Effective Temperature for Emergence (TEFFMX) in degrees Celsius
# Assumed it to be the same as for sugarcane, which is a closely related C4 crop (see sugarcane.yaml)
TEFFMX = 30.0

# Temperature Sum from Sowing to Emergence (TSUMEM) in degree days
# Total Growing Degree Days taken from Luisa Trindade & Francesco Pancaldi's parameters for Miscanthus in AquaCrop (another simpler crop growth model)
# GDD = 2000.0
#
TSUMEM = 170.0 # similar to sugarcane very high
#
# now remaining GDD = 2000 - 170 = 1830

In [28]:
# Phenological Development 
# For Parameters that are the identical for all other C4 crops, and have no external sources to indicate it should be different for Miscanthus ...
# ... I will assume the same values | These will be indicated by # C4 in the comments next to the defined variables

# IDSL, indicates wether pre-anthesis development depends on temperature (=0), plus daylength (=1), plus vernalization (=2)
IDSL = 0    # C4

# Optimum Daylength for Development in hours
DLO = 1.0   # C4

# Critical Daylength (lower threshold)
DLC = 0.0   # C4

# Growing Degree Days = 2000.0, taken from AquaCrop parameters
# Due to similar plant characteristics, I will take the same fraction of growing degree days between the two TSUM1 & TSUM2 parameters of sugarcane
# The respective fractions will be multiplied by the total growing degree days of Miscanthus Sinensis specifically
#
# Hereby for sugarcane TSUM1 = 5681 & TSUM2 = 642, thus TSUM1 = 5681 / (5681 + 642) * 1830.0 = 1644.2   & TSUM2 = 642 / (5681 + 642) * 1830.0 = 185.8
#
# Temperature Sum from Emergence to Anthesis in degree days
TSUM1 = 1644.2

# Temperature Sum from Anthesis to Maturity in degree days
TSUM2 = 185.8

# Daily Increase in Temperature Sum as a function of Average Temperature in degrees Celsius
DTSMTB = [ 0.0,  0.0,
           6.0,  0.0,
          35.0, 29.0,
          45.0, 29.0] # AquaCrop Basetemperature for Miscanthus, filled in by Francesco Pancaldi & Luisa Trindade (WUR)

# Initial Development Stage 
DVSI = 0

# Development Stage at Harvest (=2.0 at maturity)
DVSEND = 2.0

# Base Vernalization Requirement in days
VERNBASE = 14.0 # C4

# Saturated Vernalization Requirement in days
VERNSAT = 70.0 # C4

# Temperature Response Function for Vernalization in degrees Celsius
VERNRTB = [-8.0, 0.0,
           -4.0, 0.0,
            3.0, 1.0,
           10.0, 1.0,
           17.0, 0.0,
           20.0, 0.0]

In [29]:
# Initial States
# 
# Initial Total Crop Dry Weight in kg/ha
TDWI = 600.0 # Taken from sugarcane, since they have similar characteristics that the other C4 crops lack
                # These include that after harvest, a lot of biomass remains in the fields, in the form of roots and rhizomes
                # This is so since sugarcane and miscanthus are both perennial crops, which means that they have exstensive root systems 
                # These roots and rhizomes are not harvested, but they do contribute to the total dry weight of the crop, 
                # and they do contribute to the growth of the crop in the next season, since they store carbohydrates that can be used for growth in the next season

# Maximum Relative Increase in LAI per day
RGRLAI = 0.0500 # C4

In [30]:
# Crop Green Area > Not Possible Currently : WAIT for unit scaling measurement
#
# Specific Leaf Area as function of DVS
SLATB = [0.00, 0.00090,
         0.21, 0.00190,
         0.29, 0.00120,
         0.64, 0.00080,
         1.00, 0.00100,
         2.00, 0.00100]

# Specific Pod Area 
SPA = 0.0000 # C4

# Specific Stem area as function of DVS
# Take Average Stem Diameter of 8 Genotypes in 2016, and Calculate the Average Stem Area : 30.15 cm^2
# Since SSATB expects Stem Area in ha.kg^-1, we need to convert the units of the average stem area from cm^2 to ha, and then divide by the stem dry weight of the crop in kg/ha
# 30.15 cm^2 / 1 ha = 30.15 / 100 000 000 = 0.0000003015 ha
# Then we divide this by the mean stem dry weight of the 8 genotypes in kg/ha, which is 301.7 g, thus 0.3017 kg
# Hereby we get 0.0000003015 ha / 0.3017 kg = 0.000000998 ha.kg^-1
SSATB = [0.0, 0.0,
         2.0, 0.000000998]

# Life Span of Leaves Growing at 35 degrees Celsius in days
SPAN = 75.0 # Sugarcane

# Lower Threshold Temperature for Ageing of Leaves in degrees Celsius
TBASE = 10.0 # C4 majority

In [31]:
# CO2 Assimilation 
#
# Extinction Coefficient for Diffuse Visible Light as function of DVS
KDIFTB = [0.00, 0.600,
          2.00, 0.600]  # C4 median

# Initial Light-Use Efficiency Single Leaf as function of Daily Mean Temperature in degrees Celsius
EFFTB = [ 0.0, 0.500,
         40.0, 0.500]   # C4

# Maximum Leaf CO2 Assimilation Rate as function of DVS 
AMAXTB = [0.00, 70.00,
          0.14, 70.00,
          0.82, 42.00,
          2.00, 30.00]  # Sugarcane

# CO2 Level at which AMAX and EFF were measured in PPM
REFCO2L = 360.0

# Reduction Factor of AMAX as function of Average Daytime (*not* daily) Temperature in degrees Celsius
TMPFTB = [ 0.0, 0.000,
           8.0, 0.000,
          20.0, 1.000,
          35.0, 1.000,
          45.0, 0.000] # Sugarcane

# Reduction Factor of Gross Assimilation Rate as function of Low Minimum Temperature in degrees Celsius
TMNFTB = [ 5.0, 0.000,
          12.0, 1.000]

In [32]:
# Conversion Efficiency of Assimilates
# All C4 crops have similar conversion efficiencies, thus it will be assumed Miscanthus also has it somewhere in this range
# 
# Efficiency of Conversion into Leaves
CVL = 0.720     #C4

# Efficiency of Conversion into Storage Organs
CVO = 0.730    #C4 median

# Efficiency of Conversion into Roots
CVR = 0.720    #C4 

# Efficiency of Conversion into Stems
CVS = 0.720    #sugarcane, the other C4's have smaller efficiencies for stem growth 

In [33]:
# Respiration   
# From the C4 crops, maize, sorghum and sugarcane have nearly the same respiration parameters,
# thus, due to lack of specific information for miscanthus, I will assume the same values 
# 
# Relative Increase in Respiration Rate per 10 degrees Celsius temperature increase
Q10 = 2.00  #C4

# Relative Maintenance Respiration Rate of Leaves (day-1)
RML = 0.0300   #C4

# Relative Maintenance Respiration Rate of Storage Organs (day-1)
RMO = 0.0100   #C4

# Relative Maintenance Respiration Rate of Roots (day-1)
RMR = 0.0100   #C4

# Relative Maintenance Respiration Rate of Stems (day-1)
RMS = 0.0150   # Maize & Sorghum

# Reduction Factor for Senesence as function of DVS
RFSETB = [0.00, 1.000,
          2.00, 1.000]

In [34]:
# Partitioning
# Copy from sugarcane due to similar characteristics between the two crops, and due to lack of data for miscanthus
# Further use end of season/harvest stem-weight ratio & leaf-weight ratio for validation of the partitioning parameters
#
# Fraction of Total Dry Matter to Roots as Function of DVS
FRTB = [0.000, 0.670,
        0.030, 0.670,
        0.150, 0.670,
        0.230, 0.160,
        0.480, 0.160,
        1.000, 0.160,
        1.100, 0.160,
        2.000, 0.160]

# Fraction of Total Dry Matter to Leaves as Function of DVS
FLTB = [0.000, 1.000,
        0.030, 1.000,
        0.150, 0.790,
        0.230, 0.660,
        0.480, 0.240,
        1.000, 0.240,
        1.100, 0.000,
        2.000, 0.000]

# Fraction of Total Dry Matter to Stems as Function of DVS
FSTB = [0.000, 0.000,
        0.030, 0.000,
        0.150, 0.210,
        0.230, 0.340,
        0.480, 0.760,
        1.000, 0.760,
        1.100, 0.000,
        2.000, 0.000]

# Fraction of Total Dry Matter to Storage Organs as Function of DVS
FOTB = [0.000, 0.000,
        1.000, 0.000,
        1.100, 1.000,
        2.000, 1.000]

In [35]:
# Death Rates
# Same parameters chosen as for the rest of C4 crops in PCSE 

# Maximum Relative Death Rate of Leaves due to Water Stress
PERDL = 0.030

# Relative Death Rate of Stems as function of DVS
RDRRTB = [0.00, 0.000,
          1.50, 0.000,
          1.51, 0.020,
          2.00, 0.020]

# Relative Death Rate of Roots as function of DVS
RDRSTB = [0.00, 0.000,
          1.50, 0.000,
          1.51, 0.020,
          2.00, 0.020]

In [36]:
# Water Use
# Same parameters chosen as the rest of C4 crops in PCSE, eg. sugarcane, maize, millet, sorghum, etc.
# Use water use efficiency of AquaCrop as validation

# Correction Factor Transpiration Rate 
CFET = 1.2  # AquaCrop mid season crop coefficient (Kc) stage 

# Crop Group Number for Soil Water Depletion
DEPNR = 5.0     # Sugarcane & Sorghum in Group 5 | Maize & Millet in Group 4.5

# Air Ducts in Roots Present (1) or Not (0)
IAIRDU = 0

# Oxygen Stress Effect Enabled (1) or Not (0)
IOX = 0 

In [37]:
# Rooting Depth
# 
# Initial Rooting Depth in cm
RDI = 10.0  #C4

# Maximum Daily Increase in Rooting Depth in cm/day
RRI = 1.20  #C4

# Maximum Rooting Depth in cm
RDMCR = 200.0 # sugarcane 

In [38]:
# Generation of YAML file for Miscanthus parameters
miscanthus_crop = f"""# Crop parameter file for use with the PCSE implementations of the WOFOST crop simulation model
#
# Crop: Miscanthus Sinensis
# Creation date: 2026-XX-XX
#
# Derived by Paulian Oprescu (BSc Thesis, Earth Informatics Group, Wageningen Environmental Research)
# Supervised by Dr. Allard de Wit
#
# Sources:
# See companion notebook: miscanthus_parameterization.ipynb
#
Version: 1.0.0
CropParameters:
    GenericC3: &GenericC3
        CO2EFFTB:
            - {CO2EFFTB_C3}
            - multiplication factor for EFF to account for an increasing CO2 concentration
            - ['PPM', '-']
        CO2TRATB:
            - {CO2TRATB_C3}
            - multiplication factor for maximum transpiration rate TRAMX to account for an increasing CO2 concentration
            - ['PPM', '-']
        CO2AMAXTB:
            - {CO2AMAXTB_C3}
            - multiplication factor for AMAX to account for an increasing CO2 concentration
            - ['PPM', '-']
    GenericC4: &GenericC4
        CO2EFFTB:
            - {CO2EFFTB_C4}
            - multiplication factor for EFF to account for an increasing CO2 concentration
            - ['PPM', '-']
        CO2TRATB:
            - {CO2TRATB_C4}
            - multiplication factor for maximum transpiration rate TRAMX to account for an increasing CO2 concentration
            - ['PPM', '-']
        CO2AMAXTB:
            - {CO2AMAXTB_C4}
            - multiplication factor for AMAX to account for an increasing CO2 concentration
            - ['PPM', '-']
    EcoTypes:
        miscanthus: &miscanthus
            <<: *GenericC4
            #
            # EMERGENCE
            #
            TBASEM:
            - {TBASEM}
            - Lower threshold temperature for emergence
            - ['C']
            TEFFMX:
            - {TEFFMX}
            - Maximum effective temperature for emergence
            - ['C']
            TSUMEM:
            - {TSUMEM}
            - Temperature sum from sowing to emergence
            - ['C.d']
            #
            # PHENOLOGICAL DEVELOPMENT
            #
            IDSL:
            - {IDSL}
            - Indicates whether pre-anthesis development depends on temperature (=0), plus daylength (=1), plus vernalization (=2)
            - ['NA']
            DLO:
            - {DLO}
            - Optimum daylength for development
            - ['hr']
            DLC:
            - {DLC}
            - Critical daylength (lower threshold)
            - ['hr']
            TSUM1:
            - {TSUM1}
            - Temperature sum from emergence to anthesis
            - ['C.d']
            TSUM2:
            - {TSUM2}
            - Temperature sum from anthesis to maturity
            - ['C.d']
            DTSMTB:
            - {DTSMTB}
            - Daily increase in temperature sum as a function of average temperature
            - ['C', 'C']
            DVSI:
            - {DVSI}
            - Initial development stage
            - ['-']
            DVSEND:
            - {DVSEND}
            - Development stage at harvest (=2.0 at maturity)
            - ['-']
            VERNBASE:
            - {VERNBASE}
            - Base vernalization requirement 
            - ['d']
            VERNSAT:
            - {VERNSAT}
            - Saturated vernalization requirement
            - ['d']
            VERNRTB:
            - {VERNRTB}
            - Temperature response function for vernalization
            - ['C', '-']
            #
            # INITIAL STATES
            #
            TDWI:
            - {TDWI}
            - Initial total crop dry weight
            - ['kg.ha-1']
            RGRLAI:
            - {RGRLAI}
            - Maximum relative increase in LAI
            - ['d-1']
            #
            # CROP GREEN AREA
            #
            SLATB:
            - {SLATB}
            - Specific leaf area as function of DVS
            - ['-', 'ha.kg-1']
            SPA:
            - {SPA}
            - Specific pod area
            - ['ha.kg-1']
            SSATB:
            - {SSATB}
            - Specific stem area as function of DVS
            - ['-', 'ha.kg-1']
            SPAN:
            - {SPAN}
            - Life span of leaves growing at 35 degrees Celsius
            - ['d']
            TBASE:
            - {TBASE}
            - Lower threshold temperature for ageing of leaves
            - ['C']
            #
            # CO2 ASSIMILATION
            #
            KDIFTB:
            - {KDIFTB}
            - Extinction coefficient for diffuse visible light as function of DVS
            - ['-', '-']
            EFFTB:
            - {EFFTB}
            - Initial light-use efficiency single leaf as function of daily mean temperature
            - ['C', 'kg.ha-1.hr-1,J-1.m2.s1']
            AMAXTB:
            - {AMAXTB}
            - Maximum leaf CO2 assimilation rate as function of DVS
            - ['-', 'kg.ha-1.hr-1']
            REFCO2L:
            - {REFCO2L}
            - CO2 level at which AMAX and EFF were measured
            - ['PPM']
            TMPFTB:
            - {TMPFTB}
            - Reduction factor of AMAX as function of average daytime temperature
            - ['C', '-']
            TMNFTB:
            - {TMNFTB}
            - Reduction factor of gross assimilation rate as function of low minimum temperature
            - ['C', '-']
            #
            # CONVERSION EFFICIENCY OF ASSIMILATES
            #
            CVL:
            - {CVL}
            - Efficiency of conversion into leaves
            - ['mass.mass-1']
            CVO:
            - {CVO}
            - Efficiency of conversion into storage organs
            - ['mass.mass-1']
            CVR:
            - {CVR}
            - Efficiency of conversion into roots
            - ['mass.mass-1']
            CVS:
            - {CVS}
            - Efficiency of conversion into stems
            - ['mass.mass-1']
            #
            # RESPIRATION
            #
            Q10:
            - {Q10}
            - Relative increase in respiration rate per 10 degrees Celsius temperature increase
            - ['-']
            RML:
            - {RML}
            - Relative maintenance respiration rate of leaves
            - ['d-1']
            RMO:
            - {RMO}
            - Relative maintenance respiration rate of storage organs
            - ['d-1']
            RMR:
            - {RMR}
            - Relative maintenance respiration rate of roots
            - ['d-1']
            RMS:
            - {RMS}
            - Relative maintenance respiration rate of stems
            - ['d-1']
            RFSETB:
            - {RFSETB}
            - Reduction factor for senescence as function of DVS
            - ['-', '-']
            #
            # PARTITIONING
            #
            FRTB:
            - {FRTB}
            - Fraction of total dry matter to roots as function of DVS
            - ['-', 'mass.mass-1']
            FLTB:
            - {FLTB}
            - Fraction of total dry matter to leaves as function of DVS
            - ['-', 'mass.mass-1']
            FSTB:
            - {FSTB}
            - Fraction of total dry matter to stems as function of DVS
            - ['-', 'mass.mass-1']
            FOTB:
            - {FOTB}
            - Fraction of total dry matter to storage organs as function of DVS
            - ['-', 'mass.mass-1']
            #
            # DEATH RATES
            #
            PERDL:
            - {PERDL}
            - Maximum relative death rate of leaves due to water stress
            - ['-']
            RDRRTB:
            - {RDRRTB}
            - Relative death rate of stems as function of DVS
            - ['-', 'kg.kg-1.d-1']
            RDRSTB:
            - {RDRSTB}
            - Relative death rate of roots as function of DVS
            - ['-', 'kg.kg-1.d-1']
            #
            # WATER USE
            #
            CFET:
            - {CFET}
            - Correction factor transpiration rate
            - ['-']
            DEPNR:
            - {DEPNR}
            - Crop group number for soil water depletion
            - ['-']
            IAIRDU:
            - {IAIRDU}
            - Air ducts in roots present (1) or not (0)
            - ['-']
            IOX:
            - {IOX}
            - Oxygen stress effect enabled (1) or not (0)
            - ['-']
            #
            # ROOTING DEPTH
            #
            RDI:
            - {RDI}
            - Initial rooting depth
            - ['cm']
            RRI:
            - {RRI}
            - Maximum daily increase in rooting depth
            - ['cm.d-1']
            RDMCR:
            - {RDMCR}
            - Maximum rooting depth
            - ['cm']
"""

In [40]:
# Write output
os.makedirs(output_dir, exist_ok=True)
outpath = os.path.join(output_dir, f"miscanthus.yaml")
with open(outpath, 'w') as f:
    f.write(miscanthus_crop)
print(f"Miscanthus parameter file written to: {outpath}")

Miscanthus parameter file written to: /Users/paulianoprescu/Projects/wofost_miscanthus/miscanthus_calibration/input_params/miscanthus/miscanthus.yaml
